# E-Commerce Catalogue Analysis

This notebook prepares and examines the combined product catalogue through data review, cleaning, derived fields, statistical analysis, and visual exploration.

## Stage 1: Import and inspect the source data

**Aim:** Load the analysis tools and dataset, then review its initial layout.

**Activities:**
- Load pandas, NumPy, Matplotlib, and Seaborn
- Read the combined catalogue
- Review the row-column count and field names

In [ ]:
# Load the libraries used throughout the analysis
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Expand notebook display settings for easier inspection
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

# Configure a consistent appearance for charts
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

print("Libraries imported successfully!")

In [ ]:
# Read the catalogue file into a DataFrame
file_path = 'combined_dataset/Combined_dataset.csv'
df = pd.read_csv(file_path)

print(f"Dataset loaded successfully!")
print(f"\nDataset Shape: {df.shape}")
print(f"Rows: {df.shape[0]}, Columns: {df.shape[1]}")
print(f"\n{'='*50}")
print("Column Names:")
print(f"{'='*50}")
for i, col in enumerate(df.columns, 1):
    print(f"{i}. {col}")

In [ ]:
# Preview the first few product rows
print("First 10 Rows of the Dataset:")
print("="*50)
df.head()

## Stage 2: Profile the dataset

**Aim:** Establish the field types, data gaps, and descriptive statistics.

**Activities:**
- Inspect each column's data type
- Count null entries
- Use `.info()` and `.describe()` to profile the data

In [ ]:
# Review the inferred type of every field
print("Data Types of All Columns:")
print("="*50)
df.dtypes

In [ ]:
# Count missing entries by column
print("\nMissing Values Analysis:")
print("="*50)
missing_values = df.isnull().sum()
missing_percentage = (missing_values / len(df)) * 100
missing_df = pd.DataFrame({
    'Column': df.columns,
    'Missing_Count': missing_values.values,
    'Missing_Percentage': missing_percentage.values
})
missing_df = missing_df[missing_df['Missing_Count'] > 0].sort_values('Missing_Count', ascending=False)
print(missing_df.to_string(index=False))

In [ ]:
# Display the DataFrame structure and completeness
print("\nDataset Information:")
print("="*50)
df.info()

In [ ]:
# Generate descriptive numeric measures
print("\nSummary Statistics:")
print("="*50)
df.describe()

## Stage 3: Prepare reliable analysis data

**Aim:** Convert price fields, address incomplete records, and eliminate repeated products.

**Activities:**
- Transform price-related fields into numbers
- Treat missing values with appropriate rules
- Retain only unique product records

In [ ]:
# Work on a separate copy of the raw catalogue
df_clean = df.copy()

print("Data Cleaning Process:")
print("="*50)
print(f"Original dataset shape: {df_clean.shape}")

# Inspect the original and final price fields before conversion
print("\nBefore Conversion:")
print(f"initial_price dtype: {df_clean['initial_price'].dtype}")
print(f"final_price dtype: {df_clean['final_price'].dtype}")
print(f"Sample initial_price values: {df_clean['initial_price'].head()}")
print(f"Sample final_price values: {df_clean['final_price'].head()}")

In [ ]:
# Transform price fields into numeric values
# Strip formatting characters before casting to float
df_clean['initial_price'] = pd.to_numeric(df_clean['initial_price'], errors='coerce')
df_clean['final_price'] = df_clean['final_price'].astype(str).str.replace('₹', '').str.replace(',', '').str.replace('"', '')
df_clean['final_price'] = pd.to_numeric(df_clean['final_price'], errors='coerce')
df_clean['discount'] = pd.to_numeric(df_clean['discount'], errors='coerce')
df_clean['rating'] = pd.to_numeric(df_clean['rating'], errors='coerce')
df_clean['ratings_count'] = pd.to_numeric(df_clean['ratings_count'], errors='coerce')

print("\nAfter Conversion:")
print(f"initial_price dtype: {df_clean['initial_price'].dtype}")
print(f"final_price dtype: {df_clean['final_price'].dtype}")
print(f"Sample initial_price values: {df_clean['initial_price'].head()}")
print(f"Sample final_price values: {df_clean['final_price'].head()}")

In [ ]:
# Apply the missing-value treatment rules
print("\nHandling Missing Values:")
print("="*50)
print(f"Before: {df_clean.isnull().sum().sum()} total missing values")

# Use median values to complete numeric gaps
numeric_columns = ['initial_price', 'final_price', 'discount', 'rating', 'ratings_count']
for col in numeric_columns:
    if df_clean[col].isnull().sum() > 0:
        df_clean[col].fillna(df_clean[col].median(), inplace=True)

print(f"After: {df_clean.isnull().sum().sum()} total missing values")

# Drop records missing essential fields
df_clean = df_clean.dropna(subset=['product_id', 'title', 'category'])

print(f"Dataset shape after handling missing values: {df_clean.shape}")

In [ ]:
# Identify and remove repeated records
print("\nRemoving Duplicate Records:")
print("="*50)
print(f"Before: {len(df_clean)} rows")

# Use the product identifier as the uniqueness key
df_clean = df_clean.drop_duplicates(subset=['product_id'], keep='first')

print(f"After removing duplicates: {len(df_clean)} rows")
print(f"Duplicates removed: {df.shape[0] - df_clean.shape[0]} rows")
print(f"\n✓ Data Cleaning Complete!")
print(f"Final dataset shape: {df_clean.shape}")

## Stage 4: Derive analysis features

**Aim:** Build additional fields that make comparison and segmentation easier.

**Activities:**
- Calculate the difference between listed and final prices
- Combine rating and review volume into a popularity indicator
- Create useful groups from existing attributes

In [ ]:
# Build the derived analytical fields
print("Feature Engineering:")
print("="*50)

# 1. Savings between the original and final price
df_clean['price_difference'] = df_clean['initial_price'] - df_clean['final_price']
df_clean['price_diff_percentage'] = (df_clean['price_difference'] / df_clean['initial_price'] * 100).round(2)

print("\n1. Price Difference Features Created:")
print(f"   - price_difference: Absolute difference")
print(f"   - price_diff_percentage: Percentage difference")
print(f"\n   Sample values:")
print(df_clean[['initial_price', 'final_price', 'price_difference', 'price_diff_percentage']].head())

In [ ]:
# 2. Combined popularity score
# Scale ratings and review counts before combining them
# The score weights ratings at 40% and review volume at 60%

from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
df_clean['rating_normalized'] = scaler.fit_transform(df_clean[['rating']])
df_clean['ratings_count_normalized'] = scaler.fit_transform(df_clean[['ratings_count']])

# Calculate the weighted popularity indicator
df_clean['popularity_metric'] = (df_clean['rating_normalized'] * 0.4 + 
                                   df_clean['ratings_count_normalized'] * 0.6).round(3)

print("\n2. Popularity Metric Created:")
print(f"   Formula: (normalized_rating * 0.4) + (normalized_ratings_count * 0.6)")
print(f"   Range: 0 to 1 (higher = more popular)")
print(f"\n   Sample values:")
print(df_clean[['rating', 'ratings_count', 'popularity_metric']].head(10))

In [ ]:
# 3. Create price bands for simpler comparison
df_clean['price_range'] = pd.cut(df_clean['final_price'], 
                                  bins=[0, 500, 1000, 2000, 5000, 10000, float('inf')],
                                  labels=['Budget', 'Economy', 'Mid-Range', 'Premium', 'Luxury', 'Ultra-Luxury'])

# Group products by discount level
df_clean['discount_category'] = pd.cut(df_clean['discount'], 
                                       bins=[-1, 10, 20, 30, 50, 100],
                                       labels=['Low', 'Medium', 'High', 'Very High', 'Extreme'])

print("\n3. Additional Features Created:")
print(f"   - price_range: Categorized price brackets")
print(f"   - discount_category: Categorized discount levels")
print(f"\n   Distribution of price_range:")
print(df_clean['price_range'].value_counts().sort_index())
print(f"\n   Distribution of discount_category:")
print(df_clean['discount_category'].value_counts().sort_index())

In [ ]:
# Review the newly added fields
print("\n" + "="*50)
print("✓ Feature Engineering Complete!")
print("="*50)
print("\nNew Features Summary:")
print(df_clean[['price_difference', 'price_diff_percentage', 'popularity_metric', 
                'price_range', 'discount_category']].describe())

## Stage 5: Examine patterns and relationships

**Aim:** Study individual fields, relationships between fields, and category-level behaviour.

**Activities:**
- Describe individual variables
- Test relationships between selected measures
- Compare results across product categories

In [ ]:
# Single-variable exploration
print("="*70)
print("UNIVARIATE ANALYSIS: Single Variable Analysis")
print("="*70)

# Rating distribution
print("\n1. RATING ANALYSIS:")
print(f"   Mean Rating: {df_clean['rating'].mean():.2f}")
print(f"   Median Rating: {df_clean['rating'].median():.2f}")
print(f"   Std Dev: {df_clean['rating'].std():.2f}")
print(f"   Min: {df_clean['rating'].min():.2f}, Max: {df_clean['rating'].max():.2f}")
print(f"   Rating Distribution:")
print(df_clean['rating'].value_counts().head(10).sort_index())

# Review-count distribution
print("\n2. RATINGS COUNT ANALYSIS:")
print(f"   Mean Reviews: {df_clean['ratings_count'].mean():.0f}")
print(f"   Median Reviews: {df_clean['ratings_count'].median():.0f}")
print(f"   Std Dev: {df_clean['ratings_count'].std():.0f}")

# Final-price distribution
print("\n3. FINAL PRICE ANALYSIS:")
print(f"   Mean Price: ₹{df_clean['final_price'].mean():.2f}")
print(f"   Median Price: ₹{df_clean['final_price'].median():.2f}")
print(f"   Std Dev: ₹{df_clean['final_price'].std():.2f}")
print(f"   Min: ₹{df_clean['final_price'].min():.2f}, Max: ₹{df_clean['final_price'].max():.2f}")

# Discount distribution
print("\n4. DISCOUNT ANALYSIS:")
print(f"   Mean Discount: {df_clean['discount'].mean():.2f}%")
print(f"   Median Discount: {df_clean['discount'].median():.2f}%")
print(f"   Std Dev: {df_clean['discount'].std():.2f}%")
print(f"   Min: {df_clean['discount'].min():.2f}%, Max: {df_clean['discount'].max():.2f}%")

# Savings distribution
print("\n5. PRICE DIFFERENCE ANALYSIS:")
print(f"   Mean Savings: ₹{df_clean['price_difference'].mean():.2f}")
print(f"   Median Savings: ₹{df_clean['price_difference'].median():.2f}")
print(f"   Total Potential Savings: ₹{df_clean['price_difference'].sum():.0f}")

In [ ]:
# Two-variable relationship analysis
print("\n" + "="*70)
print("BIVARIATE ANALYSIS: Relationship between Variables")
print("="*70)

# 1. Relationship between rating and review count
correlation = df_clean['rating'].corr(df_clean['ratings_count'])
print(f"\n1. Rating vs Ratings Count Correlation: {correlation:.4f}")

# 2. Relationship between final price and rating
correlation = df_clean['final_price'].corr(df_clean['rating'])
print(f"2. Final Price vs Rating Correlation: {correlation:.4f}")

# 3. Relationship between discount and review count
correlation = df_clean['discount'].corr(df_clean['ratings_count'])
print(f"3. Discount vs Ratings Count Correlation: {correlation:.4f}")

# 4. Relationship between savings and rating
correlation = df_clean['price_difference'].corr(df_clean['rating'])
print(f"4. Price Difference vs Rating Correlation: {correlation:.4f}")

# 5. Relationship between popularity and final price
correlation = df_clean['popularity_metric'].corr(df_clean['final_price'])
print(f"5. Popularity Metric vs Final Price Correlation: {correlation:.4f}")

# Build a correlation matrix for the selected numeric fields
numeric_features = ['rating', 'ratings_count', 'final_price', 'discount', 
                    'price_difference', 'popularity_metric']
correlation_matrix = df_clean[numeric_features].corr()
print(f"\n\nCorrelation Matrix of Key Features:")
print(correlation_matrix)

In [ ]:
# Product-category comparison
print("\n" + "="*70)
print("CATEGORY-LEVEL ANALYSIS: Insights by Product Category")
print("="*70)

category_analysis = df_clean.groupby('category').agg({
    'final_price': ['mean', 'median', 'min', 'max'],
    'rating': ['mean', 'count'],
    'discount': 'mean',
    'ratings_count': 'mean',
    'popularity_metric': 'mean'
}).round(2)

category_analysis.columns = ['Avg_Price', 'Median_Price', 'Min_Price', 'Max_Price', 
                             'Avg_Rating', 'Product_Count', 'Avg_Discount', 
                             'Avg_Reviews', 'Avg_Popularity']

print("\nTop 10 Categories by Product Count:")
top_categories = df_clean['category'].value_counts().head(10)
print(top_categories)

print("\nCategory Performance Summary:")
print(category_analysis.sort_values('Avg_Popularity', ascending=False).head(10))

In [ ]:
# Compare results across price bands
print("\n" + "-"*70)
print("ANALYSIS BY PRICE RANGE:")
print("-"*70)

price_range_analysis = df_clean.groupby('price_range').agg({
    'final_price': ['mean', 'count'],
    'rating': 'mean',
    'discount': 'mean',
    'ratings_count': 'mean',
    'popularity_metric': 'mean'
}).round(2)

print("\nPrice Range Performance:")
print(price_range_analysis)

# Compare results across discount bands
print("\n" + "-"*70)
print("ANALYSIS BY DISCOUNT CATEGORY:")
print("-"*70)

discount_analysis = df_clean.groupby('discount_category').agg({
    'final_price': 'mean',
    'rating': 'mean',
    'product_id': 'count',
    'ratings_count': 'mean',
    'popularity_metric': 'mean'
}).round(2)

discount_analysis.columns = ['Avg_Price', 'Avg_Rating', 'Product_Count', 'Avg_Reviews', 'Avg_Popularity']
print("\nDiscount Category Performance:")
print(discount_analysis)

## Stage 6: Present the findings visually

**Aim:** Use clear charts to communicate distributions, comparisons, and associations.

**Charts:**
- Histograms for numeric distributions
- Bar charts for categorical comparisons
- Box plots for spread and outlier review

In [ ]:
# Visual analysis
print("="*70)
print("CREATING VISUALIZATIONS")
print("="*70)

# Apply the chart style settings
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# 1. Histograms for key numeric fields
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('Distribution of Key Numeric Variables', fontsize=16, fontweight='bold')

# Ratings
axes[0, 0].hist(df_clean['rating'], bins=30, color='skyblue', edgecolor='black', alpha=0.7)
axes[0, 0].set_title('Distribution of Ratings', fontweight='bold')
axes[0, 0].set_xlabel('Rating')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].axvline(df_clean['rating'].mean(), color='red', linestyle='--', label=f'Mean: {df_clean["rating"].mean():.2f}')
axes[0, 0].legend()

# Review counts
axes[0, 1].hist(df_clean['ratings_count'], bins=50, color='lightgreen', edgecolor='black', alpha=0.7)
axes[0, 1].set_title('Distribution of Ratings Count', fontweight='bold')
axes[0, 1].set_xlabel('Number of Reviews')
axes[0, 1].set_ylabel('Frequency')
axes[0, 1].axvline(df_clean['ratings_count'].mean(), color='red', linestyle='--', label=f'Mean: {df_clean["ratings_count"].mean():.0f}')
axes[0, 1].legend()

# Final prices
axes[0, 2].hist(df_clean['final_price'], bins=50, color='lightcoral', edgecolor='black', alpha=0.7)
axes[0, 2].set_title('Distribution of Final Prices', fontweight='bold')
axes[0, 2].set_xlabel('Price (₹)')
axes[0, 2].set_ylabel('Frequency')
axes[0, 2].axvline(df_clean['final_price'].mean(), color='red', linestyle='--', label=f'Mean: ₹{df_clean["final_price"].mean():.0f}')
axes[0, 2].legend()

# Discounts
axes[1, 0].hist(df_clean['discount'], bins=40, color='lightyellow', edgecolor='black', alpha=0.7)
axes[1, 0].set_title('Distribution of Discounts', fontweight='bold')
axes[1, 0].set_xlabel('Discount (%)')
axes[1, 0].set_ylabel('Frequency')
axes[1, 0].axvline(df_clean['discount'].mean(), color='red', linestyle='--', label=f'Mean: {df_clean["discount"].mean():.2f}%')
axes[1, 0].legend()

# Price savings
axes[1, 1].hist(df_clean['price_difference'], bins=40, color='lightblue', edgecolor='black', alpha=0.7)
axes[1, 1].set_title('Distribution of Price Differences', fontweight='bold')
axes[1, 1].set_xlabel('Price Difference (₹)')
axes[1, 1].set_ylabel('Frequency')
axes[1, 1].axvline(df_clean['price_difference'].mean(), color='red', linestyle='--', label=f'Mean: ₹{df_clean["price_difference"].mean():.0f}')
axes[1, 1].legend()

# Popularity scores
axes[1, 2].hist(df_clean['popularity_metric'], bins=40, color='plum', edgecolor='black', alpha=0.7)
axes[1, 2].set_title('Distribution of Popularity Metric', fontweight='bold')
axes[1, 2].set_xlabel('Popularity Score (0-1)')
axes[1, 2].set_ylabel('Frequency')
axes[1, 2].axvline(df_clean['popularity_metric'].mean(), color='red', linestyle='--', label=f'Mean: {df_clean["popularity_metric"].mean():.3f}')
axes[1, 2].legend()

plt.tight_layout()
plt.show()
print("✓ Histograms created successfully!")

In [ ]:
# 2. Box plots for spread and unusual values
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('Box Plots for Outlier Detection', fontsize=16, fontweight='bold')

# Ratings box plot
sns.boxplot(y=df_clean['rating'], ax=axes[0, 0], color='skyblue')
axes[0, 0].set_title('Rating Distribution', fontweight='bold')
axes[0, 0].set_ylabel('Rating')

# Review-count box plot
sns.boxplot(y=df_clean['ratings_count'], ax=axes[0, 1], color='lightgreen')
axes[0, 1].set_title('Ratings Count Distribution', fontweight='bold')
axes[0, 1].set_ylabel('Number of Reviews')

# Final-price box plot
sns.boxplot(y=df_clean['final_price'], ax=axes[0, 2], color='lightcoral')
axes[0, 2].set_title('Final Price Distribution', fontweight='bold')
axes[0, 2].set_ylabel('Price (₹)')

# Discount box plot
sns.boxplot(y=df_clean['discount'], ax=axes[1, 0], color='lightyellow')
axes[1, 0].set_title('Discount Distribution', fontweight='bold')
axes[1, 0].set_ylabel('Discount (%)')

# Savings box plot
sns.boxplot(y=df_clean['price_difference'], ax=axes[1, 1], color='lightblue')
axes[1, 1].set_title('Price Difference Distribution', fontweight='bold')
axes[1, 1].set_ylabel('Price Difference (₹)')

# Popularity-score box plot
sns.boxplot(y=df_clean['popularity_metric'], ax=axes[1, 2], color='plum')
axes[1, 2].set_title('Popularity Metric Distribution', fontweight='bold')
axes[1, 2].set_ylabel('Popularity Score (0-1)')

plt.tight_layout()
plt.show()
print("✓ Box plots created successfully!")

In [ ]:
# 3. Bar charts for categorical comparisons

# Ten largest categories by listing count
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle('Category-Level Insights', fontsize=16, fontweight='bold')

# Leading categories by listing count
top_categories = df_clean['category'].value_counts().head(10)
axes[0, 0].barh(range(len(top_categories)), top_categories.values, color='steelblue')
axes[0, 0].set_yticks(range(len(top_categories)))
axes[0, 0].set_yticklabels(top_categories.index)
axes[0, 0].set_title('Top 10 Categories by Product Count', fontweight='bold')
axes[0, 0].set_xlabel('Number of Products')
axes[0, 0].invert_yaxis()

# Leading categories by mean rating
top_categories_rating = df_clean.groupby('category')['rating'].mean().nlargest(10)
axes[0, 1].barh(range(len(top_categories_rating)), top_categories_rating.values, color='lightgreen')
axes[0, 1].set_yticks(range(len(top_categories_rating)))
axes[0, 1].set_yticklabels(top_categories_rating.index)
axes[0, 1].set_title('Top 10 Categories by Average Rating', fontweight='bold')
axes[0, 1].set_xlabel('Average Rating')
axes[0, 1].invert_yaxis()

# Product count by price band
price_range_counts = df_clean['price_range'].value_counts()
axes[1, 0].bar(range(len(price_range_counts)), price_range_counts.values, color='coral')
axes[1, 0].set_xticks(range(len(price_range_counts)))
axes[1, 0].set_xticklabels(price_range_counts.index, rotation=45)
axes[1, 0].set_title('Products by Price Range', fontweight='bold')
axes[1, 0].set_ylabel('Number of Products')

# Product count by discount band
discount_counts = df_clean['discount_category'].value_counts()
axes[1, 1].bar(range(len(discount_counts)), discount_counts.values, color='mediumpurple')
axes[1, 1].set_xticks(range(len(discount_counts)))
axes[1, 1].set_xticklabels(discount_counts.index, rotation=45)
axes[1, 1].set_title('Products by Discount Category', fontweight='bold')
axes[1, 1].set_ylabel('Number of Products')

plt.tight_layout()
plt.show()
print("✓ Category bar charts created successfully!")

In [ ]:
# 4. Correlation heat map
plt.figure(figsize=(10, 8))
correlation_matrix = df_clean[['rating', 'ratings_count', 'final_price', 'discount', 
                                'price_difference', 'popularity_metric']].corr()
sns.heatmap(correlation_matrix, annot=True, fmt='.3f', cmap='coolwarm', center=0, 
            square=True, linewidths=1, cbar_kws={"shrink": 0.8})
plt.title('Correlation Matrix of Key Features', fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()
print("✓ Correlation heatmap created successfully!")

In [ ]:
# 5. Scatter plots for selected relationships
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle('Bivariate Relationships', fontsize=16, fontweight='bold')

# Rating compared with review count
axes[0, 0].scatter(df_clean['ratings_count'], df_clean['rating'], alpha=0.5, s=30, color='blue')
axes[0, 0].set_xlabel('Number of Reviews')
axes[0, 0].set_ylabel('Rating')
axes[0, 0].set_title('Rating vs Ratings Count', fontweight='bold')
axes[0, 0].grid(True, alpha=0.3)

# Final price compared with rating
axes[0, 1].scatter(df_clean['final_price'], df_clean['rating'], alpha=0.5, s=30, color='green')
axes[0, 1].set_xlabel('Final Price (₹)')
axes[0, 1].set_ylabel('Rating')
axes[0, 1].set_title('Final Price vs Rating', fontweight='bold')
axes[0, 1].grid(True, alpha=0.3)

# Discount compared with rating
axes[1, 0].scatter(df_clean['discount'], df_clean['rating'], alpha=0.5, s=30, color='red')
axes[1, 0].set_xlabel('Discount (%)')
axes[1, 0].set_ylabel('Rating')
axes[1, 0].set_title('Discount vs Rating', fontweight='bold')
axes[1, 0].grid(True, alpha=0.3)

# Popularity score compared with final price
axes[1, 1].scatter(df_clean['final_price'], df_clean['popularity_metric'], alpha=0.5, s=30, color='purple')
axes[1, 1].set_xlabel('Final Price (₹)')
axes[1, 1].set_ylabel('Popularity Metric (0-1)')
axes[1, 1].set_title('Popularity Metric vs Final Price', fontweight='bold')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
print("✓ Scatter plots created successfully!")

In [ ]:
# 6. Box plots across groups
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Analysis by Price Range and Discount Category', fontsize=16, fontweight='bold')

# Ratings across price bands
sns.boxplot(x='price_range', y='rating', data=df_clean, ax=axes[0], palette='Set2')
axes[0].set_title('Rating Distribution by Price Range', fontweight='bold')
axes[0].set_xlabel('Price Range')
axes[0].set_ylabel('Rating')
axes[0].tick_params(axis='x', rotation=45)

# Ratings across discount bands
sns.boxplot(x='discount_category', y='rating', data=df_clean, ax=axes[1], palette='Set3')
axes[1].set_title('Rating Distribution by Discount Category', fontweight='bold')
axes[1].set_xlabel('Discount Category')
axes[1].set_ylabel('Rating')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()
print("✓ Box plots by category created successfully!")

In [ ]:
# 7. Consolidated statistics table
print("\n" + "="*70)
print("ANALYSIS SUMMARY")
print("="*70)

print(f"\nDataset Overview:")
print(f"  Total Products: {len(df_clean):,}")
print(f"  Total Categories: {df_clean['category'].nunique()}")
print(f"  Total Unique Brands: {df_clean['seller_name'].nunique() if 'seller_name' in df_clean.columns else 'N/A'}")
print(f"  Average Product Rating: {df_clean['rating'].mean():.2f}/5.0")
print(f"  Average Discount: {df_clean['discount'].mean():.2f}%")
print(f"  Total Potential Savings: ₹{df_clean['price_difference'].sum():,.0f}")
print(f"  Average Customer Reviews per Product: {df_clean['ratings_count'].mean():.0f}")

print(f"\nKey Insights:")
print(f"  1. Most Popular Category: {df_clean['category'].value_counts().idxmax()}")
print(f"     (with {df_clean['category'].value_counts().max()} products)")
print(f"  2. Highest Rated Category: {df_clean.groupby('category')['rating'].mean().idxmax()}")
print(f"     (with {df_clean.groupby('category')['rating'].mean().max():.2f} average rating)")
print(f"  3. Highest Average Discount: {df_clean.groupby('category')['discount'].mean().idxmax()}")
print(f"     (with {df_clean.groupby('category')['discount'].mean().max():.2f}% average discount)")
print(f"  4. Most Reviewed Category: {df_clean.groupby('category')['ratings_count'].mean().idxmax()}")
print(f"     (with {df_clean.groupby('category')['ratings_count'].mean().max():.0f} average reviews)")

print("\n✓ All visualizations created successfully!")

## Closing notes

### Work completed

The notebook completes the following work:

1. **Import and review**: Loads the combined catalogue and profiles its structure.
2. **Data quality checks**: Reviews field types, missing entries, and descriptive measures.
3. **Preparation**: Standardizes prices, resolves incomplete values, and removes duplicates.
4. **Derived fields**: Adds savings, popularity, price-band, and discount-band variables.
5. **Analysis**: Investigates individual measures, relationships, and category differences.
6. **Communication**: Creates histograms, box plots, bar charts, heat maps, and scatter plots.

### Areas examined

- **Catalogue mix**: Product groups vary in the number and type of listings.
- **Customer feedback**: Rating patterns can be compared by price and discount groups.
- **Pricing**: The data shows both price variation and the amount saved from listed prices.
- **Engagement**: The popularity score combines feedback quality with feedback volume.
- **Category behaviour**: Pricing, discounting, and engagement differ across categories.

### Possible follow-up work

- Use the results to support category-specific pricing decisions.
- Test discount levels against the popularity measures.
- Prioritize marketing experiments for promising categories.
- Refresh the analysis when newer catalogue data becomes available.